# Analyze Your Own Molecules

Now it's your turn! In this capstone notebook, you'll:

1. **Choose** molecules from PubChem or enter your own SMILES
2. **Compute** their molecular descriptors
3. **Cluster** them alongside the 30-compound reference library
4. **Discover** where your molecules fall in chemical space

This parallels the Birdsong capstone (record your own birds) and the Climate capstone (choose your own weather station).

## Step 1: Import Libraries

In [ ]:
!pip install rdkit-pypi pubchempy -q

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs, Draw
from IPython.display import display
import pubchempy as pcp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## Step 2: Choose Your Molecules

### Option A: Search PubChem by name
Enter compound names and we'll fetch their SMILES automatically.

### Option B: Enter SMILES directly
If you know the SMILES notation, enter them directly.

### Need ideas? Try these:
| Category | Examples |
|----------|---------|
| Painkillers | morphine, codeine, tramadol |
| Vitamins | ascorbic acid, riboflavin, thiamine |
| Flavors | vanillin, menthol, cinnamaldehyde |
| Amino acids | glycine, alanine, tryptophan |
| Solvents | acetone, dichloromethane, tetrahydrofuran |

In [ ]:
# ====== OPTION A: Search by name ======
my_compound_names = [
    "vanillin",
    "menthol",
    "cinnamaldehyde",
    "limonene",
    "eugenol",
]

skip_pubchem = False

# ====== OPTION B: Enter SMILES directly ======
# Uncomment and use this instead if you have SMILES:
# my_compounds = [
#     {"name": "My Molecule 1", "smiles": "CCO"},
#     {"name": "My Molecule 2", "smiles": "c1ccccc1"},
# ]
# skip_pubchem = True

In [ ]:
# Fetch from PubChem
if not skip_pubchem:
    my_compounds = []
    print("Fetching from PubChem...")
    for name in my_compound_names:
        try:
            results = pcp.get_compounds(name, "name")
            if results:
                c = results[0]
                my_compounds.append({
                    "name": name.capitalize(),
                    "smiles": c.canonical_smiles,
                })
                print(f"  {name}: {c.canonical_smiles}")
        except Exception as e:
            print(f"  {name}: FAILED ({e})")

print(f"\nFetched {len(my_compounds)} compounds")

## Step 3: Visualize Your Molecules

In [ ]:
mol_objs = [Chem.MolFromSmiles(c["smiles"]) for c in my_compounds]
mol_objs = [m for m in mol_objs if m is not None]
mol_names = [c["name"] for c in my_compounds]

if mol_objs:
    img = Draw.MolsToGridImage(mol_objs, molsPerRow=3, subImgSize=(300, 250),
                                legends=mol_names[:len(mol_objs)])
    display(img)
else:
    print("No valid molecules to display.")

## Step 4: Load Reference Library

The same 32 compounds from Notebook 3.

In [ ]:
reference_library = [
    {"name": "Methanol",       "smiles": "CO",                     "family": "Alcohol"},
    {"name": "Ethanol",        "smiles": "CCO",                    "family": "Alcohol"},
    {"name": "1-Propanol",     "smiles": "CCCO",                   "family": "Alcohol"},
    {"name": "1-Butanol",      "smiles": "CCCCO",                  "family": "Alcohol"},
    {"name": "1-Pentanol",     "smiles": "CCCCCO",                 "family": "Alcohol"},
    {"name": "Isopropanol",    "smiles": "CC(C)O",                 "family": "Alcohol"},
    {"name": "Cyclohexanol",   "smiles": "OC1CCCCC1",              "family": "Alcohol"},
    {"name": "Formic Acid",    "smiles": "O=CO",                   "family": "Acid"},
    {"name": "Acetic Acid",    "smiles": "CC(=O)O",                "family": "Acid"},
    {"name": "Propionic Acid", "smiles": "CCC(=O)O",               "family": "Acid"},
    {"name": "Butyric Acid",   "smiles": "CCCC(=O)O",              "family": "Acid"},
    {"name": "Pentanoic Acid", "smiles": "CCCCC(=O)O",             "family": "Acid"},
    {"name": "Benzoic Acid",   "smiles": "OC(=O)c1ccccc1",         "family": "Acid"},
    {"name": "Benzene",        "smiles": "c1ccccc1",               "family": "Aromatic"},
    {"name": "Toluene",        "smiles": "Cc1ccccc1",              "family": "Aromatic"},
    {"name": "Naphthalene",    "smiles": "c1ccc2ccccc2c1",         "family": "Aromatic"},
    {"name": "Aniline",        "smiles": "Nc1ccccc1",              "family": "Aromatic"},
    {"name": "Phenol",         "smiles": "Oc1ccccc1",              "family": "Aromatic"},
    {"name": "Styrene",        "smiles": "C=Cc1ccccc1",            "family": "Aromatic"},
    {"name": "Anthracene",     "smiles": "c1ccc2cc3ccccc3cc2c1",   "family": "Aromatic"},
    {"name": "Methylamine",    "smiles": "CN",                     "family": "Amine"},
    {"name": "Ethylamine",     "smiles": "CCN",                    "family": "Amine"},
    {"name": "Dimethylamine",  "smiles": "CNC",                    "family": "Amine"},
    {"name": "Trimethylamine", "smiles": "CN(C)C",                 "family": "Amine"},
    {"name": "Propylamine",    "smiles": "CCCN",                   "family": "Amine"},
    {"name": "Diethylamine",   "smiles": "CCNCC",                  "family": "Amine"},
    {"name": "Aspirin",        "smiles": "CC(=O)Oc1ccccc1C(=O)O",  "family": "Drug"},
    {"name": "Ibuprofen",      "smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "family": "Drug"},
    {"name": "Acetaminophen",  "smiles": "CC(=O)Nc1ccc(O)cc1",     "family": "Drug"},
    {"name": "Caffeine",       "smiles": "Cn1c(=O)c2c(ncn2C)n(C)c1=O", "family": "Drug"},
    {"name": "Naproxen",       "smiles": "COc1ccc2cc(ccc2c1)C(C)C(=O)O", "family": "Drug"},
    {"name": "Lidocaine",      "smiles": "CCN(CC)CC(=O)Nc1c(C)cccc1C", "family": "Drug"},
]

print(f"Reference library: {len(reference_library)} compounds")

## Step 5: Compute Features for All Compounds

In [ ]:
def compute_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=512)
    fp_arr = np.zeros(512)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    desc = [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol), Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol), Descriptors.FractionCSP3(mol),
        mol.GetNumHeavyAtoms(), mol.GetNumBonds(),
        Descriptors.RingCount(mol), Descriptors.BertzCT(mol),
        Descriptors.LabuteASA(mol), Descriptors.PEOE_VSA1(mol),
        Descriptors.Chi0v(mol), Descriptors.Kappa1(mol),
    ]
    return fp_arr, np.array(desc)

# Process all compounds
all_fps, all_descs, all_names, all_families = [], [], [], []

for c in reference_library:
    result = compute_features(c["smiles"])
    if result:
        all_fps.append(result[0])
        all_descs.append(result[1])
        all_names.append(c["name"])
        all_families.append(c["family"])

for c in my_compounds:
    result = compute_features(c["smiles"])
    if result:
        all_fps.append(result[0])
        all_descs.append(result[1])
        all_names.append(c["name"])
        all_families.append("YOUR COMPOUND")

# PCA on fingerprints
pca = PCA(n_components=10)
fp_pca = pca.fit_transform(np.array(all_fps))
feature_matrix = np.hstack([fp_pca, np.array(all_descs)])

print(f"Feature matrix: {feature_matrix.shape[0]} compounds x {feature_matrix.shape[1]} features")
print(f"Your compounds: {sum(1 for f in all_families if f == 'YOUR COMPOUND')}")

## Step 6: Train the Autoencoder

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_matrix)
X_tensor = torch.FloatTensor(X_scaled)

class MoleculeAutoencoder(nn.Module):
    def __init__(self, input_dim=26, bottleneck=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

torch.manual_seed(42)
model = MoleculeAutoencoder()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

print("Training...")
for epoch in range(5000):
    out = model(X_tensor)
    loss = nn.MSELoss()(out, X_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f"  Epoch {epoch}: loss = {loss.item():.6f}")
print(f"  Final: {loss.item():.6f}")

## Step 7: Where Do Your Molecules Fall?

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model.encoder(X_tensor).numpy()

family_colors = {
    "Alcohol": "#3498db", "Acid": "#e74c3c", "Aromatic": "#9b59b6",
    "Amine": "#2ecc71", "Drug": "#f39c12", "YOUR COMPOUND": "gold",
}

plt.figure(figsize=(14, 10))

# Plot reference compounds
for i, name in enumerate(all_names):
    fam = all_families[i]
    if fam == "YOUR COMPOUND":
        continue
    color = family_colors.get(fam, "gray")
    plt.scatter(embeddings[i, 0], embeddings[i, 1], c=color, s=60,
                zorder=3, edgecolors="black", linewidth=0.5, alpha=0.6)
    plt.annotate(name, (embeddings[i, 0], embeddings[i, 1]),
                 textcoords="offset points", xytext=(5, 5), fontsize=6, alpha=0.6)

# Plot YOUR compounds as big stars
for i, name in enumerate(all_names):
    if all_families[i] == "YOUR COMPOUND":
        plt.scatter(embeddings[i, 0], embeddings[i, 1],
                    c="gold", s=350, marker="*", zorder=5,
                    edgecolors="black", linewidth=1.5)
        plt.annotate(f"  {name}", (embeddings[i, 0], embeddings[i, 1]),
                     fontsize=11, fontweight="bold", color="darkred")

for fam, color in family_colors.items():
    marker = "*" if fam == "YOUR COMPOUND" else "o"
    s = 200 if fam == "YOUR COMPOUND" else 60
    plt.scatter([], [], c=color, s=s, marker=marker, label=fam, edgecolors="black")
plt.legend(title="Category", loc="best")

plt.title("Your Molecules in Chemical Space")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

## Step 8: Interpretation

Look at where your stars landed:
- **Which reference compounds are they closest to?**
- **Which chemical family do they most resemble?**
- **Do your molecules cluster together or spread out?**
- **Does the clustering match the chemistry you know?**

## Bonus: Save to Google Drive

In [ ]:
# Uncomment to save results:
# from google.colab import drive
# drive.mount('/content/drive')
#
# results = pd.DataFrame({
#     "name": all_names,
#     "family": all_families,
#     "dim1": embeddings[:, 0],
#     "dim2": embeddings[:, 1],
# })
# results.to_csv('/content/drive/MyDrive/my_chemistry_results.csv', index=False)
# print("Saved!")

## Congratulations!

You've completed the full FAU Chemistry & AI curriculum! Across 6 notebooks, you learned:

1. **Python fundamentals** — variables, lists, loops, functions in chemistry context
2. **Molecular representation** — SMILES, RDKit, molecular descriptors, fingerprints
3. **Visual pattern recognition** — structure grids, descriptor profiles, chemical space plots
4. **Machine learning** — autoencoders for unsupervised clustering of molecules
5. **3D visualization & dynamics** — py3Dmol, PubChem API, chemical kinetics ODEs
6. **Independent research** — choosing and analyzing your own molecules

These are the same tools and techniques used by working computational chemists and drug discovery scientists. Keep exploring!